# Tutorial 3: Multi-Asset Portfolio Training

## Why Multi-Asset Matters

Single-asset trading is a solved toy problem. Real alpha comes from **portfolio allocation** — deciding how much capital to put in each asset, when to rotate between them, and how to exploit correlation structure.

Spectral-Env-Core supports N simultaneous assets with:
- Per-asset drift, volatility, and price dynamics
- Cholesky-correlated paths (assets move together realistically)
- A single continuous action vector controlling all positions simultaneously

The agent must learn not just *when to trade* but *what to trade* — a fundamentally harder and more realistic problem.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spectral_env_core import SpectralTradingEnv

In [ ]:
# 3-asset portfolio: Tech, Energy, Bonds (different risk profiles)
env = SpectralTradingEnv(
    num_assets=3,
    num_steps=252,
    initial_price=[180.0, 95.0, 110.0],     # Different price levels
    drift=[0.20, 0.12, 0.04],               # Tech high growth, bonds low
    volatility=[0.30, 0.25, 0.08],          # Tech volatile, bonds stable
    garch_alpha=0.08,
    garch_beta=0.85,
    jump_intensity=1.5,
    jump_mean=-0.02,
    jump_std=0.05,
    starting_cash=100_000,
    max_shares=500,
    max_trade_size=50,
    # Cross-correlation matrix
    correlation=np.array([
        [1.00, 0.45, -0.20],   # Tech-Energy: moderate positive
        [0.45, 1.00,  0.05],   # Energy-Bonds: near zero
        [-0.20, 0.05, 1.00],   # Tech-Bonds: negative (flight to safety)
    ]),
)

print(f"Action space: {env.action_space}")
print(f"  → Agent outputs 3 continuous values in [-1, 1]")
print(f"  → Each controls buy/sell for one asset independently")
print(f"\nObservation dim: {env.observation_space.shape[0]}")
print(f"  → 3 × 30 price windows + 3 shares + cash + portfolio + exit_cost + time_remaining")

In [ ]:
# Visualise correlated paths
env.reset(seed=42)

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
labels = ['Tech (σ=30%)', 'Energy (σ=25%)', 'Bonds (σ=8%)']
colors = ['tab:blue', 'tab:orange', 'tab:green']

for i in range(3):
    axes[i].plot(env.brownian_path[:, i], color=colors[i], lw=1.5)
    axes[i].set_ylabel(f'{labels[i]}\nPrice ($)')
    axes[i].grid(True, alpha=0.3)

axes[0].set_title('3-Asset Correlated Price Paths (ρ_tech_bonds = -0.20)', fontsize=12)
axes[2].set_xlabel('Trading Day')
plt.tight_layout()
plt.show()

print("Notice: when Tech drops sharply, Bonds tend to rise (negative correlation).")
print("An agent that learns this relationship can hedge by rotating into bonds during stress.")

## The Correlation Challenge

Correlation means the agent can't treat each asset independently. A selloff in Tech propagates partially to Energy (ρ=0.45) but drives money into Bonds (ρ=-0.20). The optimal strategy depends on the *joint* dynamics, not individual assets.

This is why the `SpectralExtractor` in our training pipeline uses a **shared encoder** across all asset price windows — the network learns to recognise cross-asset patterns, not just per-asset signals.

### What the agent can learn:
- **Risk-off rotation**: when vol spikes (GARCH), shift from Tech → Bonds
- **Momentum capture**: in bull regime, overweight high-drift Tech
- **Diversification**: maintain partial positions in negatively correlated assets
- **Relative value**: when Tech/Energy spread widens beyond normal, mean-revert it

In [ ]:
# Show the correlation is preserved in generated data
n_episodes = 500
returns_all = []

for i in range(n_episodes):
    env.reset(seed=i)
    log_ret = np.diff(np.log(env.brownian_path), axis=0)
    returns_all.append(log_ret)

all_returns = np.vstack(returns_all)  # (n_episodes * 252, 3)
empirical_corr = np.corrcoef(all_returns.T)

print("Target correlation matrix:")
print(np.array2string(np.array([[1.00, 0.45, -0.20], [0.45, 1.00, 0.05], [-0.20, 0.05, 1.00]]), precision=3))
print(f"\nEmpirical correlation (from {n_episodes} episodes):")
print(np.array2string(empirical_corr, precision=3))
print(f"\n→ The generator preserves the target correlation structure accurately.")

## Training a Multi-Asset Agent

The action space is `Box([-1, 1], shape=(3,))` — three continuous values controlling simultaneous positions across all assets. The agent must learn portfolio construction, not just timing.

The alpha-relative reward (vs. equal-weight buy-and-hold benchmark) ensures the agent is rewarded for *outperforming passive allocation*, not just riding the market up.

---

**Next tutorial:** [04 — Crypto & Fractional Positions](04_crypto_fractional.ipynb) — trade BTC and ETH with dollar-based position sizing.